# Pipeline 4 - Metrics

In [17]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import rioxarray as xrio

from swot_toolkit.pipe2 import open_output_dir
from swot_toolkit.planetary import parse_s2_id

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
OUTPUT_DIR = Path("/data/swot/output/curua-una/")

In [10]:
aoi, s2_ids = open_output_dir(OUTPUT_DIR)
s2_ids

Reading KML file: /data/swot/output/curua-una/kml/Curua-Una.kml


{'2025-08-14': 'S2B_MSIL2A_20250818T135709_R067_T21MYS_20250818T172904',
 '2024-09-24': 'S2A_MSIL2A_20240927T135701_R067_T21MYS_20240927T190725'}

In [11]:
MOSAIC_DATE = "2024-09-24"

In [22]:
output_dir = OUTPUT_DIR
s2_id = s2_ids[MOSAIC_DATE]
s2_meta = parse_s2_id(s2_id)

## Open opera mask

In [23]:
opera_mask_path = next((output_dir / "opera").glob(f"*{s2_meta['sensing_date']}*.tif"))
opera_mask = xrio.open_rasterio(opera_mask_path).squeeze()
opera_mask

<xarray.DataArray (y: 1975, x: 1917)> Size: 4MB
[3786075 values with dtype=uint8]
Coordinates:
    band         int64 8B 1
  * x            (x) float64 15kB 7.498e+05 7.499e+05 ... 8.073e+05 8.073e+05
  * y            (y) float64 16kB -3.021e+05 -3.022e+05 ... -3.614e+05
    spatial_ref  int64 8B 0
Attributes: (12/49)
    ACCODE:                                                                  ...
    AEROSOL_CLASS_REMAPPING_ENABLED:                                         ...
    AEROSOL_NOT_WATER_TO_HIGH_CONF_WATER_FMASK_VALUES:                       ...
    AEROSOL_PARTIAL_SURFACE_AGGRESSIVE_TO_HIGH_CONF_WATER_FMASK_VALUES:      ...
    AEROSOL_PARTIAL_SURFACE_WATER_CONSERVATIVE_TO_HIGH_CONF_WATER_FMASK_VALUE...
    AEROSOL_WATER_MODERATE_CONF_TO_HIGH_CONF_WATER_FMASK_VALUES:             ...
    ...                                                                          ...
    WORLDCOVER_COVERAGE:                                                     ...
    WORLDCOVER_SOURCE:                                                       ...
    AREA_OR_POINT:                                                           ...
    _FillValue:                                                              ...
    scale_factor:                                                            ...
    add_offset:                                                              ...

## Open reference mask

In [26]:

ref_mask_path = next((output_dir / "ref_masks").glob(f"*{s2_meta['sensing_date']}*.tif"))

ref_mask = xrio.open_rasterio(ref_mask_path).squeeze()
ref_mask = ref_mask.rio.reproject_match(opera_mask)
ref_mask

<xarray.DataArray (y: 1975, x: 1917)> Size: 4MB
array([[0, 0, 0, ..., 0, 0, 2],
       [0, 0, 0, ..., 0, 0, 2],
       [0, 0, 0, ..., 0, 0, 2],
       ...,
       [0, 0, 0, ..., 0, 0, 2],
       [0, 0, 0, ..., 0, 0, 2],
       [0, 0, 0, ..., 0, 0, 2]], shape=(1975, 1917), dtype=uint8)
Coordinates:
    band         int64 8B 1
    spatial_ref  int64 8B 0
  * x            (x) float64 15kB 7.498e+05 7.499e+05 ... 8.073e+05 8.073e+05
  * y            (y) float64 16kB -3.021e+05 -3.022e+05 ... -3.614e+05
Attributes:
    long_name:      Classification
    AREA_OR_POINT:  Area
    scale_factor:   1.0
    add_offset:     0.0
    _FillValue:     2

In [29]:
opera_mask.data[opera_mask.data == 2] = 1
opera_mask.data[opera_mask.data > 200] = 3

np.unique(opera_mask)

array([0, 1, 3], dtype=uint8)

In [34]:
from swot_toolkit.metrics import calc_metrics

results = calc_metrics(ref_mask, opera_mask, ["iou", "f1", "precision", "recall", "coverage", "water_coverage"])

In [36]:
import pandas as pd
pd.DataFrame.from_dict(results, orient="index", columns=["value"])

,value
iou,0.9108
f1,0.9533
precision,0.9998
recall,0.9110
coverage,0.9111
water_coverage,0.9769


In [28]:
np.unique(opera_mask)

array([  0,   1,   2, 252, 253, 255], dtype=uint8)